In [1]:
!pip install duckdb

In [2]:
import duckdb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

conn = duckdb.connect('movies.db')
conn.execute("INSTALL vss; LOAD vss;")

model = SentenceTransformer('all-MiniLM-L6-v2')

# Fetch all movies
movies = conn.execute("SELECT id, title, summary FROM movies").fetchall()

# Get existing IDs to skip
existing = {row[0] for row in conn.execute("SELECT movie_id FROM movie_embeddings").fetchall()}

# Process with progress bar
for movie_id, title, summary in tqdm(movies, desc="Generating embeddings"):
    if movie_id in existing:
        continue
    text = f"{title} {summary or ''}"
    embedding = model.encode(text, normalize_embeddings=True)
    conn.execute(
        "INSERT INTO movie_embeddings (movie_id, embedding) VALUES (?, ?)",
        [movie_id, embedding.tolist()]
    )

conn.commit()
print(f"Inserted {len(movies) - len(existing)} new embeddings")

/home/arthur/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 647.15it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Generating embeddings: 100%|██████████| 51332/51332 [00:00<00:00, 658758.42it/s]

Inserted 0 new embeddings


In [5]:
print(
    conn.execute("""
        PRAGMA table_info(movie_embeddings)
    """).fetchall()
)
print(
    conn.execute("""
        DESCRIBE movie_embeddings
    """).fetchall()
)

[(0, 'movie_id', 'BIGINT', True, None, True), (1, 'embedding', 'FLOAT[384]', False, None, False)]
[('movie_id', 'BIGINT', 'NO', 'PRI', None, None), ('embedding', 'FLOAT[384]', 'YES', None, None, None)]


In [7]:
row = conn.execute("""
    SELECT embedding
    FROM movie_embeddings_new
    LIMIT 1
""").fetchone()

print(repr(row[0]))
print(type(row[0]))

'[-0.0640200600028038, -0.03520666062831879, 0.0629686638712883, -0.061003006994724274, 0.005614461377263069, 0.022223277017474174, 0.03572627156972885, 0.03491000458598137, 0.03499115630984306, 0.04565432295203209, 0.08071878552436829, -0.033863745629787445, -0.03601302579045296, -0.04027920216321945, 0.058194562792778015, -0.019922465085983276, 0.05453795567154884, 0.0072431908920407295, -0.008534982800483704, 0.0011416920460760593, -0.02853933349251747, -0.04218560457229614, 0.046590909361839294, 0.02291131019592285, -0.05372755229473114, 0.035386886447668076, 0.061619844287633896, -0.026114845648407936, -0.09275147318840027, -0.058019716292619705, -0.02985096350312233, -0.031513042747974396, -0.07473982870578766, 0.05593410134315491, 0.021243447437882423, 0.03398022800683975, 0.014962609857320786, -0.005592102650552988, 0.0962744951248169, -0.08325478434562683, -0.05653984099626541, -0.06736679375171661, 0.0483141653239727, 0.03060814179480076, 0.0011470140889286995, 0.010261034592

In [15]:
import ast

conn.execute("""
CREATE TABLE movie_embeddings_new (
    movie_id BIGINT PRIMARY KEY,
    embedding FLOAT[384]
)
""")

rows = conn.execute("""
    SELECT movie_id, embedding
    FROM movie_embeddings
""").fetchall()

# with tqdm
for movie_id, embedding_str in tqdm(rows):
    embedding = ast.literal_eval(embedding_str)

    conn.execute("""
        INSERT INTO movie_embeddings_new
        VALUES (?, ?)
    """, [movie_id, embedding])

conn.commit()

100%|██████████| 51332/51332 [10:28:56<00:00,  1.36it/s]      


In [8]:
print(conn.execute("""
DESCRIBE movie_embeddings_new
""").fetchall())

[('movie_id', 'BIGINT', 'YES', None, None, None), ('embedding', 'VARCHAR', 'YES', None, None, None)]


In [10]:
import duckdb

print("Python package:", duckdb.__version__)
print("Engine:", conn.execute("SELECT version()").fetchone())
print("Module:", duckdb.__file__)

Python package: 1.5.3
Engine: ('v1.5.3',)
Module: /home/arthur/miniconda3/lib/python3.13/site-packages/duckdb/__init__.py


In [53]:
import duckdb

# 1. Connect and setup VSS
new_conn = duckdb.connect('movies3.db')
new_conn.execute("INSTALL vss; LOAD vss; SET hnsw_enable_experimental_persistence=true")

# 2. Attach OLD database
new_conn.execute("ATTACH 'movies.db' AS old_db6 (TYPE SQLITE)")

# 3. Query DuckDB's internal catalog for the attached DB's tables
tables = new_conn.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_catalog = 'old_db6'
""").fetchall()

# 4. Loop and migrate the tables safely
for (table_name,) in tables:
    # Skip the specific table you don't want
    if table_name == 'movie_embeddings':
        continue
    
    print(f"Migrating table: {table_name}...")
    
    # Safely wrap table names in quotes in case they contain spaces or reserved words
    new_conn.execute(f"""
        CREATE TABLE main."{table_name}" AS
        SELECT * FROM old_db6."{table_name}"
    """)

print("Migration complete!")

# 5. Migrate embeddings (parse VARCHAR → FLOAT[384])
# embeddings = new_conn.execute("SELECT movie_id, embedding FROM old_db6.movie_embeddings").fetchall()
# for movie_id, embedding_str in embeddings:
#     embedding_list = ast.literal_eval(embedding_str)
#     new_conn.execute(
#         "INSERT INTO movie_embeddings (movie_id, embedding) VALUES (?, ?)",
#         [movie_id, embedding_list]
#     )

# # 6. Create HNSW index
# new_conn.execute("""
#     CREATE INDEX movie_embeddings_idx ON movie_embeddings
#     USING HNSW (embedding)
#     WITH (metric='cosine')
# """)

new_conn.commit()
print("Migration complete! Use movies3.db")

Migrating table: movies...
Migrating table: genres...
Migrating table: movie_genres...
Migrating table: reviews...
Migrating table: keywords...
Migrating table: movie_keywords...
Migrating table: people...
Migrating table: movie_people...
Migrating table: similar_movies...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Migrating table: recommended_movies...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Migrating table: watch_providers...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Migrating table: spoken_languages...
Migrating table: movie_spoken_languages...
Migrating table: processed_pages...
Migrating table: movie_virtues...
Migrating table: movie_virtues_fastopic...
Migrating table: movie_virtues_bertopic...
Migrating table: movie_scores...
Migrating table: movie_virtue_scores...
Migrating table: movie_virtue_scores_wide...
Migrating table: movie_embeddings_new...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Migrating table: movie_embeddings2...
Migrating table: test_vectors...
Migrating table: my_vector_table...
Migration complete!
Migration complete! Use movies3.db


In [13]:
import duckdb

# 1. Connect and setup VSS
new_conn = duckdb.connect('movies.db')

new_conn.execute("DROP TABLE IF EXISTS my_vector_table")

new_conn.execute("""
CREATE TABLE my_vector_table (vec FLOAT[3])
""")



new_conn.execute("""
INSERT INTO my_vector_table
    SELECT array_value(a, b, c)
    FROM range(1, 10) ra(a), range(1, 10) rb(b), range(1, 10) rc(c);
""")

print(new_conn.execute("DESCRIBE my_vector_table").fetchall())
print(new_conn.execute("SELECT * FROM my_vector_table").fetchall())

print(new_conn.execute("""
SELECT *
FROM my_vector_table
ORDER BY array_distance(vec, [1, 2, 3]::FLOAT[3])
LIMIT 3;
""").fetchall())

[('vec', 'FLOAT[3]', 'YES', None, None, None)]
[((1.0, 1.0, 1.0),), ((1.0, 2.0, 1.0),), ((1.0, 3.0, 1.0),), ((1.0, 4.0, 1.0),), ((1.0, 5.0, 1.0),), ((1.0, 6.0, 1.0),), ((1.0, 7.0, 1.0),), ((1.0, 8.0, 1.0),), ((1.0, 9.0, 1.0),), ((1.0, 1.0, 2.0),), ((1.0, 2.0, 2.0),), ((1.0, 3.0, 2.0),), ((1.0, 4.0, 2.0),), ((1.0, 5.0, 2.0),), ((1.0, 6.0, 2.0),), ((1.0, 7.0, 2.0),), ((1.0, 8.0, 2.0),), ((1.0, 9.0, 2.0),), ((1.0, 1.0, 3.0),), ((1.0, 2.0, 3.0),), ((1.0, 3.0, 3.0),), ((1.0, 4.0, 3.0),), ((1.0, 5.0, 3.0),), ((1.0, 6.0, 3.0),), ((1.0, 7.0, 3.0),), ((1.0, 8.0, 3.0),), ((1.0, 9.0, 3.0),), ((1.0, 1.0, 4.0),), ((1.0, 2.0, 4.0),), ((1.0, 3.0, 4.0),), ((1.0, 4.0, 4.0),), ((1.0, 5.0, 4.0),), ((1.0, 6.0, 4.0),), ((1.0, 7.0, 4.0),), ((1.0, 8.0, 4.0),), ((1.0, 9.0, 4.0),), ((1.0, 1.0, 5.0),), ((1.0, 2.0, 5.0),), ((1.0, 3.0, 5.0),), ((1.0, 4.0, 5.0),), ((1.0, 5.0, 5.0),), ((1.0, 6.0, 5.0),), ((1.0, 7.0, 5.0),), ((1.0, 8.0, 5.0),), ((1.0, 9.0, 5.0),), ((1.0, 1.0, 6.0),), ((1.0, 2.0, 6.0),), ((1.0, 3.0, 

In [71]:
import duckdb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

new_conn.execute("INSTALL vss; LOAD vss;")

model = SentenceTransformer('all-MiniLM-L6-v2')

# Fetch all movies
movies = new_conn.execute("SELECT id, title, summary FROM movies").fetchall()

# drop movie embeddings to start fresh
new_conn.execute("DROP TABLE IF EXISTS movie_embeddings")
new_conn.execute("""
CREATE TABLE movie_embeddings (
    movie_id BIGINT PRIMARY KEY,
    embedding FLOAT[384]
)
""")

def create_embedding_text(movie_tuple):
    movie_id, title, summary = movie_tuple  # Unpack tuple
    summary = summary or ''

    # Get genres
    genres = new_conn.execute("""
        SELECT g.name
        FROM movie_genres mg
        JOIN genres g ON mg.genre_id = g.id
        WHERE mg.movie_id = ?
    """, [movie_id]).fetchall()
    genre_str = ', '.join(g[0] for g in genres)

    # Get keywords
    keywords = new_conn.execute("""
        SELECT k.name
        FROM movie_keywords mk
        JOIN keywords k ON mk.keyword_id = k.id
        WHERE mk.movie_id = ?
        LIMIT 5
    """, [movie_id]).fetchall()
    keyword_str = ', '.join(k[0] for k in keywords)

    return f"Title: {title} | Summary: {summary} | Genres: {genre_str} | Keywords: {keyword_str}"

# Usage
movies = new_conn.execute("SELECT id, title, summary FROM movies").fetchall()

for movie in tqdm(movies, desc="Generating improved embeddings"):
    text = create_embedding_text(movie)
    embedding = model.encode(text, normalize_embeddings=True)
    new_conn.execute(
        "INSERT INTO movie_embeddings (movie_id, embedding) VALUES (?, ?)",
        [movie[0], embedding.tolist()]
    )

new_conn.commit()
print(f"Inserted {len(movies) - len(existing)} new embeddings")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Generating improved embeddings: 100%|██████████| 51332/51332 [13:25<00:00, 63.71it/s]

Inserted 51332 new embeddings


In [72]:
print(new_conn.execute("DESCRIBE movie_embeddings").fetchall())
print(new_conn.execute("SELECT count(*) FROM movie_embeddings limit 5").fetchall())

[('movie_id', 'BIGINT', 'NO', 'PRI', None, None), ('embedding', 'FLOAT[384]', 'YES', None, None, None)]
[(51332,)]


In [14]:
def find_similar(movie_id, limit=10):
    """Find similar movies by movie_id"""
    results = new_conn.execute("""
        SELECT
            m.id,
            m.title,
            1 - array_cosine_distance(e1.embedding, e2.embedding) AS similarity
        FROM movie_embeddings e1
        JOIN movie_embeddings e2 ON e1.movie_id != e2.movie_id
        JOIN movies m ON e2.movie_id = m.id
        WHERE e1.movie_id = ?
        ORDER BY similarity DESC
        LIMIT ?
    """, [movie_id, limit]).fetchall()
    return results

def find_similar_by_title(title, limit=10):
    """Find similar movies by movie title"""
    movie_id = new_conn.execute(
        "SELECT id FROM movies WHERE title == ?", [title]
    ).fetchone()[0]
    return find_similar(movie_id, limit)

# Example usage
similar = find_similar_by_title("Jurassic World", 50)
for movie_id, title, similarity in similar:
    print(f"{title}: {similarity:.4f}")

Jurassic World Dominion: 0.8100
Jurassic World: Fallen Kingdom: 0.7688
Jurassic World Rebirth: 0.7564
Jurassic Attack: 0.7010
Jurassic City: 0.6842
Jurassic World Camp Cretaceous: Hidden Adventure: 0.6799
Beyond Jurassic Park: 0.6797
Return to Jurassic Park: 0.6188
Dinosaur World: 0.6094
Dinosaur Island: 0.6079
Jurassic Triangle: 0.6033
The Dinosaur Project: 0.5825
My Tyrano: Together, Forever: 0.5807
Anonymous Rex: 0.5788
The Making Of  Jurassic Park III: 0.5531
The Jurassic Games: 0.5519
The Adventures of Tyrano Boy: 0.5510
LEGO Jurassic World: The Indominus Escape: 0.5383
Dinosaur Quiz: 0.5366
The Land That Time Forgot: 0.5282
Pterodactyl: 0.5259
The Eden Formula: 0.5248
The Good Dinosaur: 0.5210
Snake 3: Dinosaur vs. Python: 0.5197
Dinotopia: 0.5195
Half-Shell Heroes: Blast to the Past: 0.5170
Age of Dinosaurs: 0.5123
Poseidon Rex: 0.5107
Walking with Dinosaurs: 0.5104
Dinocroc: 0.5092
Sea Monsters: A Prehistoric Adventure: 0.5069
Triassic Attack: 0.5029
Discovering Dinotopia: 0.50

In [15]:
def hybrid_recommend(watched_movie_ids, virtue_weight=0.3, embedding_weight=0.7, limit=10):
    """
    watched_movie_ids: List of movie IDs the user watched
    virtue_weight: Weight for virtue similarity (0-1)
    embedding_weight: Weight for embedding similarity (0-1)
    limit: Number of recommendations
    """

    # === Step 1: Get candidate movies from embedding similarity ===
    candidates = set()
    for mid in watched_movie_ids:
        similar = new_conn.execute("""
            SELECT m.id, 1 - array_cosine_distance(e1.embedding, e2.embedding) AS sim
            FROM movie_embeddings e1
            JOIN movie_embeddings e2 ON e1.movie_id = ? AND e1.movie_id != e2.movie_id
            JOIN movies m ON e2.movie_id = m.id
            ORDER BY sim DESC
            LIMIT 15
        """, [mid]).fetchall()
        candidates.update((row[0], row[1]) for row in similar)

    # Remove already watched
    candidates = [c for c in candidates if c[0] not in watched_movie_ids]

    if not candidates:
        return []

    # === Step 2: Calculate average virtue scores for watched movies ===
    watched_virtues = new_conn.execute("""
        SELECT
            AVG(Wisdom),
            AVG(Courage),
            AVG(Humanity),
            AVG(Justice),
            AVG(Temperance),
            AVG(Transcendence)
        FROM movie_virtue_scores_wide
        WHERE movie_id IN ?
    """, [tuple(watched_movie_ids)]).fetchone()

    # === Step 3: Score all candidates ===
    results = []
    for candidate_id, emb_sim in candidates:
        # Get candidate's virtue scores
        cand_virtues = new_conn.execute("""
            SELECT Wisdom, Courage, Humanity, Justice, Temperance, Transcendence
            FROM movie_virtue_scores_wide
            WHERE movie_id = ?
        """, [candidate_id]).fetchone()

        if cand_virtues:
            # Cosine similarity between virtue vectors
            w_vec = [float(x) for x in watched_virtues]
            c_vec = [float(x) for x in cand_virtues]
            dot = sum(w * c for w, c in zip(w_vec, c_vec))
            norm_w = sum(x**2 for x in w_vec)**0.5
            norm_c = sum(x**2 for x in c_vec)**0.5
            virtue_sim = dot / (norm_w * norm_c) if norm_w * norm_c > 0 else 0
        else:
            virtue_sim = 0

        combined = (embedding_weight * emb_sim) + (virtue_weight * virtue_sim)
        results.append((candidate_id, combined, emb_sim, virtue_sim))

    # Sort by combined score
    results.sort(key=lambda x: -x[1])

    # Get movie details
    movie_ids = [r[0] for r in results[:limit]]
    return new_conn.execute(f"""
        SELECT id, title, vote_average
        FROM movies
        WHERE id IN {tuple(movie_ids)}
        ORDER BY CASE id { ' '.join(f'WHEN {mid} THEN {i}' for i, mid in enumerate(movie_ids)) } END
    """).fetchall()

In [16]:
# User watched these movies
watched = [11, 157336, 13475, 10681, 687163]

# Get recommendations (70% embedding, 30% virtues)
recommendations = hybrid_recommend(watched, virtue_weight=0.9, embedding_weight=0.1, limit=50)

for movie_id, title, rating in recommendations:
    print(f"{title} (Rating: {rating:.1f})")

Travelers (Rating: 4.2)
Mind Meld: Secrets Behind the Voyage of a Lifetime (Rating: 7.2)
Prometheus (Rating: 6.6)
Star Trek Into Darkness (Rating: 7.3)
Alien Expedition (Rating: 3.1)
Star Wars: Episode II - Attack of the Clones (Rating: 6.6)
The Story of Star Wars (Rating: 7.2)
Voyagers (Rating: 6.0)
Trekkies 2 (Rating: 6.3)
Lo and Behold: Reveries of the Connected World (Rating: 6.6)
Solo: A Star Wars Story (Rating: 6.6)
Star Raiders: The Adventures of Saber Raine (Rating: 3.9)
Star Trek: Of Gods and Men (Rating: 4.2)
The Creator (Rating: 7.0)
Star Wars: Episode I - The Phantom Menace (Rating: 6.6)
Robotech: The Shadow Chronicles (Rating: 6.0)
The Characters of Star Wars (Rating: 6.8)
Project Gemini (Rating: 5.7)
Elio (Rating: 6.9)
EVA (Rating: 6.4)
Star Wars: The Clone Wars (Rating: 6.1)
Star Wars: The Last Jedi (Rating: 6.8)
Interstellar: Nolan's Odyssey (Rating: 7.8)
Arrival (Rating: 7.6)
What We Left Behind: Looking Back at Star Trek: Deep Space Nine (Rating: 7.0)
The Day the Eart

In [17]:
import numpy as np

def virtue_focused_recommend(watched_movie_ids, limit=10):
    # === Step 1: User's virtue profile ===
    virtues = new_conn.execute("""
        SELECT
            AVG(Wisdom), AVG(Courage), AVG(Humanity),
            AVG(Justice), AVG(Temperance), AVG(Transcendence)
        FROM movie_virtue_scores_wide
        WHERE movie_id IN ?
    """, [tuple(int(x) for x in watched_movie_ids)]).fetchone()

    virtue_names = ['Wisdom', 'Courage', 'Humanity', 'Justice', 'Temperance', 'Transcendence']
    virtue_scores = [float(v) or 0 for v in virtues]
    top_virtues = sorted(zip(virtue_names, virtue_scores), key=lambda x: -x[1])[:2]
    print(f"User's top virtues: {[v[0] for v in top_virtues]}")

    # === Step 2: Get movies high in top virtues ===
    virtue_candidates = set()
    for virtue, _ in top_virtues:
        movies = new_conn.execute(f"""
            SELECT movie_id
            FROM movie_virtue_scores_wide
            WHERE movie_id NOT IN {tuple(int(x) for x in watched_movie_ids)}
            ORDER BY {virtue} DESC
            LIMIT 4000
        """).fetchall()
        virtue_candidates.update(int(m[0]) for m in movies)

    virtue_candidates = list(virtue_candidates)
    if not virtue_candidates:
        return []

    # === Step 3: Get embeddings ===
    watched_embs = np.array([
        list(e[0]) for e in new_conn.execute(f"""
            SELECT embedding FROM movie_embeddings
            WHERE movie_id IN {tuple(int(x) for x in watched_movie_ids)}
        """).fetchall()
    ])

    candidate_data = new_conn.execute(f"""
        SELECT movie_id, embedding FROM movie_embeddings
        WHERE movie_id IN {tuple(int(x) for x in virtue_candidates)}
    """).fetchall()
    candidate_ids = np.array([int(c[0]) for c in candidate_data])  # Convert to Python ints
    candidate_embs = np.array([list(c[1]) for c in candidate_data])

    # === Step 4: Vectorized similarity ===
    sim_matrix = watched_embs @ candidate_embs.T
    avg_sims = np.mean(sim_matrix, axis=0)

    # === Step 5: Return top results ===
    top_indices = np.argsort(-avg_sims)[:limit]
    top_ids = [int(x) for x in candidate_ids[top_indices]]  # Convert to Python ints

    return new_conn.execute(f"""
        SELECT id, title, vote_average
        FROM movies
        WHERE id IN {tuple(top_ids)}
    """).fetchall()

In [18]:
watched = [11]
recommendations = virtue_focused_recommend(watched, limit=10)
for movie_id, title, rating in recommendations:
    print(f"{title} (Rating: {rating:.1f})")

User's top virtues: ['Justice', 'Humanity']
Sakura Wars: The Movie (Rating: 4.5)
Ra.One (Rating: 5.7)
Tales of an Ancient Empire (Rating: 2.8)
Task Force 2001 (Rating: 4.2)
Ratchet & Clank (Rating: 5.8)
LEGO Hero Factory: Savage Planet (Rating: 6.3)
Persona 3 the Movie: #2 Midsummer Knight's Dream (Rating: 7.3)
Deadpool 2 (Rating: 7.5)
LEGO DC Super Hero Girls: Super-Villain High (Rating: 6.1)
Rebel Moon - Part One: A Child of Fire (Rating: 6.2)


In [24]:
import re

def responsive_recommend(
    watched_movie_ids,
    watched_ratings=None,
    liked_ids=None,
    disliked_ids=None,
    selected_virtues=None,
    limit=10,
    virtue_weight=0.95,
    embedding_weight=0.05,
    boost_factor=1.5,
    provider_ids=None,
    country_code="NL",
    explore_factor=0.0,
):
    if not watched_movie_ids:
        return []

    watched_ids = [int(x) for x in watched_movie_ids]

    # -----------------------------
    # PROVIDER FILTER
    # -----------------------------
    provider_filter_sql = ""
    provider_params = []

    if provider_ids :
        provider_filter_sql = f"""
        AND EXISTS (
            SELECT 1
            FROM watch_providers wp
            WHERE wp.movie_id = m.id
              AND wp.country_code = ?
              AND wp.provider_id IN ({",".join(["?"] * len(provider_ids))})
        )
        """
        provider_params = [country_code] + provider_ids

    # -----------------------------
    # LIKES / DISLIKES FROM RATINGS
    # -----------------------------
    if liked_ids is None:
        liked_ids = []

    if disliked_ids is None:
        disliked_ids = []

    if watched_ratings is not None:
        watched_ratings = (
            watched_ratings + [5] * len(watched_ids)
        )[:len(watched_ids)]

        for mid, rating in zip(watched_ids, watched_ratings):
            if rating >= 7 and mid not in liked_ids:
                liked_ids.append(mid)
            elif rating <= 4 and mid not in disliked_ids:
                disliked_ids.append(mid)


    if liked_ids:
        rows = new_conn.execute(f"""
            SELECT movie_id, embedding
            FROM movie_embeddings
            WHERE movie_id IN ({",".join(["?"] * len(liked_ids))})
        """, liked_ids).fetchall()

        emb_map = {mid: np.array(e) for mid, e in rows}

        liked_embs = np.array([emb_map[mid] for mid in liked_ids])

        weights = np.linspace(1.0, 3.0, len(liked_ids))
        weights /= weights.sum()

        liked_profile = np.average(liked_embs, axis=0, weights=weights)
    else:
        liked_profile = None

    liked_ids = [int(x) for x in liked_ids] if liked_ids else []
    disliked_ids = [int(x) for x in disliked_ids] if disliked_ids else []

    # -----------------------------
    # LIKED EMBEDDING PROFILE
    # -----------------------------
    if liked_ids:
        liked_embs = np.array([
            list(e[0]) for e in new_conn.execute(f"""
                SELECT embedding
                FROM movie_embeddings
                WHERE movie_id IN ({",".join(["?"] * len(liked_ids))})
            """, liked_ids).fetchall()
        ])

        weights = np.linspace(1.0, 3.0, len(liked_ids))
        weights /= weights.sum()

        liked_profile = np.average(liked_embs, axis=0, weights=weights)
    else:
        liked_profile = None

    # -----------------------------
    # VIRTUE PROFILE (BASE)
    # -----------------------------
    if liked_ids:
        liked_virtues = np.array(new_conn.execute(f"""
            SELECT AVG(Wisdom),
                   AVG(Courage),
                   AVG(Humanity),
                   AVG(Justice),
                   AVG(Temperance),
                   AVG(Transcendence)
            FROM movie_virtue_scores_wide
            WHERE movie_id IN ({",".join(["?"] * len(liked_ids))})
        """, liked_ids).fetchone())
    else:
        liked_virtues = np.zeros(6)

    # -----------------------------
    # LOAD WATCHED CLUSTERS
    # -----------------------------
    watched_clusters = new_conn.execute(f"""
        SELECT cluster_id
        FROM movie_clusters
        WHERE movie_id IN ({",".join(["?"] * len(watched_ids))})
    """, watched_ids).fetchall()

    watched_clusters = [c[0] for c in watched_clusters]
    bad_clusters = set(watched_clusters)

    # -----------------------------
    # WATCHED TITLES (FRANCHISE BLOCK)
    # -----------------------------
    watched_titles = new_conn.execute(f"""
        SELECT title
        FROM movies
        WHERE id IN ({",".join(["?"] * len(watched_ids))})
    """, watched_ids).fetchall()

    watched_titles = [t[0].lower() for t in watched_titles]

    def extract_words(text):
        return set(re.findall(r"\w+", text.lower()))

    watched_keywords = set()
    for t in watched_titles:
        watched_keywords |= extract_words(t)

    def shares_franchise(title):
        words = extract_words(title)
        return len(words & watched_keywords) >= 2

    # -----------------------------
    # CANDIDATES
    # -----------------------------
    candidates = new_conn.execute(f"""
        SELECT m.id,
               m.title,
               e.embedding,
               v.Wisdom, v.Courage, v.Humanity,
               v.Justice, v.Temperance, v.Transcendence,
               mc.cluster_id
        FROM movies m
        JOIN movie_embeddings e ON m.id = e.movie_id
        JOIN movie_virtue_scores_wide v ON m.id = v.movie_id
        JOIN movie_clusters mc ON m.id = mc.movie_id
        WHERE m.id NOT IN ({",".join(["?"] * len(watched_ids))})
          AND m.vote_average >= 7
          {provider_filter_sql}
    """, watched_ids + provider_params).fetchall()

    if not candidates:
        return []

    candidate_ids = np.array([c[0] for c in candidates])
    candidate_titles = np.array([c[1] for c in candidates])
    candidate_embs = np.array([list(c[2]) for c in candidates])
    candidate_virtues = np.array([c[3:9] for c in candidates], dtype=float)
    candidate_clusters = np.array([c[9] for c in candidates])

    # -----------------------------
    # HARD FRANCHISE FILTER
    # -----------------------------
    mask = np.array([
        not shares_franchise(t)
        for t in candidate_titles
    ])

    candidate_ids = candidate_ids[mask]
    candidate_titles = candidate_titles[mask]
    candidate_embs = candidate_embs[mask]
    candidate_virtues = candidate_virtues[mask]
    candidate_clusters = candidate_clusters[mask]

    if len(candidate_ids) == 0:
        return []

    # -----------------------------
    # EMBEDDING SCORE
    # -----------------------------
    emb_sims = np.zeros(len(candidate_ids))

    if liked_profile is not None:
        emb_sims += candidate_embs @ liked_profile

    if len(emb_sims) > 1:
        emb_sims = (emb_sims - emb_sims.min()) / (np.ptp(emb_sims) + 1e-9)

    # -----------------------------
    # VIRTUE SCORE (FIXED + STRONGER)
    # -----------------------------
    virtue_names = [
        "Wisdom", "Courage", "Humanity",
        "Justice", "Temperance", "Transcendence"
    ]

    target_virtues = liked_virtues.astype(float).copy()

    if selected_virtues:
        for i, name in enumerate(virtue_names):
            if selected_virtues.get(name, False):
                target_virtues[i] *= boost_factor
            else:
                target_virtues[i] *= 0.85

    weights = np.ones(6)
    if selected_virtues:
        for i, name in enumerate(virtue_names):
            if selected_virtues.get(name, False):
                weights[i] = boost_factor
            else:
                weights[i] = 0.8

    virtue_sims = np.array([
        np.dot(weights * liked_virtues, v * weights) /
        (np.linalg.norm(weights * liked_virtues) * np.linalg.norm(v * weights) + 1e-9)
        for v in candidate_virtues
    ])

    # stabilize + amplify signal
    virtue_sims = np.tanh(2.0 * virtue_sims)

    # -----------------------------
    # NOVELTY
    # -----------------------------
    lnorm = np.linalg.norm(liked_virtues)

    novelty = np.array([
        1 - (
            np.dot(liked_virtues, v) /
            (lnorm * np.linalg.norm(v))
        )
        if lnorm > 0 and np.linalg.norm(v) > 0 else 0
        for v in candidate_virtues
    ])

    if len(novelty) > 1:
        novelty = (novelty - novelty.min()) / (np.ptp(novelty) + 1e-9)

    # -----------------------------
    # CLUSTER HOPPING
    # -----------------------------
    cluster_penalty = np.array([
        1.0 if c not in bad_clusters else 0.2
        for c in candidate_clusters
    ])

    if explore_factor > 0.4:
        cluster_penalty = np.array([
            1.0 if c not in bad_clusters else 0.0
            for c in candidate_clusters
        ])

    # -----------------------------
    # SCORES
    # -----------------------------
    base_score = (
        embedding_weight * emb_sims +
        virtue_weight * virtue_sims
    )

    explore_score = (
        0.10 * emb_sims +
        0.25 * novelty +
        0.30 * cluster_penalty +
        0.35 * virtue_sims
    )

    combined = (
        (1 - explore_factor) * base_score +
        explore_factor * explore_score
    )

    # combined += np.random.uniform(0, 0.00, len(combined))

    # -----------------------------
    # TOP RESULTS
    # -----------------------------
    top_idx = np.argsort(-combined)[:limit]
    top_ids = candidate_ids[top_idx]

    return new_conn.execute(f"""
        SELECT id, title, vote_average
        FROM movies
        WHERE id IN ({",".join(["?"] * len(top_ids))})
    """, list(map(int, top_ids))).fetchall()

In [25]:
# Single movie
print("=== Watched: [11] ===")
for m in responsive_recommend([11], limit=5):
    print(m)

# Two movies
print("\n=== Watched: [11, 157336] ===")
for m in responsive_recommend([11, 157336], limit=5):
    print(m)

=== Watched: [11] ===
(13, 'Forrest Gump', 8.465)
(87, 'Indiana Jones and the Temple of Doom', 7.307)
(141819, 'Unforgiven', 7.0)
(876753, 'All About My Mother', 7.5)
(1265936, 'Ariel', 7.0)

=== Watched: [11, 157336] ===
(13, 'Forrest Gump', 8.465)
(87, 'Indiana Jones and the Temple of Doom', 7.307)
(141819, 'Unforgiven', 7.0)
(876753, 'All About My Mother', 7.5)
(1265936, 'Ariel', 7.0)


In [27]:
# # Pure virtue-based (no embedding influence)
res = responsive_recommend([11], virtue_weight=1, embedding_weight=0, limit=5)
print("Pure Virtue:", res)

# # Pure embedding-based
res = responsive_recommend([11], virtue_weight=0, embedding_weight=1, limit=5)
print("Pure Embedding:", res)

# Balanced
res = responsive_recommend([11, 14160, 508442], virtue_weight=0.97, embedding_weight=0.03, limit=5)
print("Balanced:", res)

Pure Virtue: [(13, 'Forrest Gump', 8.465), (87, 'Indiana Jones and the Temple of Doom', 7.307), (141819, 'Unforgiven', 7.0), (876753, 'All About My Mother', 7.5), (1265936, 'Ariel', 7.0)]
Pure Embedding: [(13, 'Forrest Gump', 8.465), (87, 'Indiana Jones and the Temple of Doom', 7.307), (141819, 'Unforgiven', 7.0), (876753, 'All About My Mother', 7.5), (1265936, 'Ariel', 7.0)]
Balanced: [(13, 'Forrest Gump', 8.465), (87, 'Indiana Jones and the Temple of Doom', 7.307), (141819, 'Unforgiven', 7.0), (876753, 'All About My Mother', 7.5), (1265936, 'Ariel', 7.0)]


In [29]:
selected_virtues = {

    "Humanity": True
}

results = responsive_recommend(
    watched_movie_ids=[11, 157336],
    watched_ratings=[0, 0],  # <-- THIS IS THE KEY
    selected_virtues=selected_virtues,

)
print("Selected Virtues Boosted:", results)

Selected Virtues Boosted: [(13, 'Forrest Gump', 8.465), (87, 'Indiana Jones and the Temple of Doom', 7.307), (154, 'Star Trek II: The Wrath of Khan', 7.45), (36380, "Boys Don't Cry", 7.416), (141819, 'Unforgiven', 7.0), (358891, 'The Promised Land', 7.0), (555582, 'The Outsiders', 7.2), (876753, 'All About My Mother', 7.5), (1243063, 'Fargo', 8.5), (1265936, 'Ariel', 7.0)]


In [32]:
selected_virtues = {
    # "Wisdom": True,
    "Humanity": True,
    "Courage": True,
    # "Justice": True,
    # "Temperance": False,
    # "Transcendence": False,
}


results = responsive_recommend(
    watched_movie_ids=[11],
    watched_ratings=[1, 10, 10],  # <-- THIS IS THE KEY
    limit=10,
    virtue_weight=0.9,
    embedding_weight=0.1,
    selected_virtues=selected_virtues,
    provider_ids=[8],  # Disney Plus
    boost_factor=2,
    explore_factor=0.3
)
print("Selected Virtues Boosted:", results)

# Print the virtues of the movies in the results to verify the boost
for movie_id, title, rating in results:
    virtues = new_conn.execute("""
        SELECT Wisdom, Courage, Humanity, Justice, Temperance, Transcendence
        FROM movie_virtue_scores_wide
        WHERE movie_id = ?
    """, [movie_id]).fetchone()
    print(f"{title} - Virtues: {virtues}")

Selected Virtues Boosted: [(1070186, 'Eko', 7.479), (1074313, 'Falling in Love Like in Movies', 8.75), (1093247, 'John Mulaney: Baby J', 7.159), (1093971, 'Srikanth', 7.558), (1098164, "Lewis Capaldi: How I'm Feeling Now", 7.6), (1101799, 'Queens on the Run', 7.022), (1103621, 'How to Make Millions Before Grandma Dies', 8.12), (1104102, 'Swing Into Romance', 7.3), (1115191, 'Poisoned: The Dirty Truth About Your Food', 7.061), (1128717, 'Unknown: The Lost Pyramid', 7.391)]
Eko - Virtues: (0.400561170041, 0.48685957196799995, 0.361404510085, 0.38088892272099995, 0.6049179286590001, 0.56024773209)
Falling in Love Like in Movies - Virtues: (0.473995401738, 0.5089972511540001, 0.62967751124, 0.523120002053, 0.607560317313, 0.554253615846)
John Mulaney: Baby J - Virtues: (0.597891609318, 0.7001558180409999, 0.611136790079, 0.693793847798, 0.714144319092, 0.6119869469870001)
Srikanth - Virtues: (0.749653170101, 0.747065772619, 0.7244162366470001, 0.735605444052, 0.7325073020840001, 0.74965058

In [2]:
import duckdb
new_conn = duckdb.connect('movies.db')

In [4]:
import numpy as np
rows = new_conn.execute("""
    SELECT movie_id, embedding
    FROM movie_embeddings
""").fetchall()

movie_ids = [r[0] for r in rows]
embeddings = np.array([np.array(r[1]) for r in rows])

embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)

k = 50

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init="auto"
)

cluster_ids = kmeans.fit_predict(embeddings)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
new_conn.execute("DROP TABLE IF EXISTS movie_clusters")

new_conn.execute("""
CREATE TABLE IF NOT EXISTS movie_clusters (
    movie_id INTEGER PRIMARY KEY,
    cluster_id INTEGER
);
""")

for mid, cid in zip(movie_ids, cluster_ids):
    new_conn.execute("""
        INSERT OR REPLACE INTO movie_clusters (movie_id, cluster_id)
        VALUES (?, ?)
    """, (mid, int(cid)))

new_conn.commit()

In [426]:
import collections

counts = collections.Counter(cluster_ids)
print(counts.most_common(100))

[(np.int32(29), 1884), (np.int32(24), 1575), (np.int32(49), 1565), (np.int32(40), 1523), (np.int32(25), 1481), (np.int32(41), 1452), (np.int32(46), 1440), (np.int32(8), 1388), (np.int32(30), 1386), (np.int32(20), 1275), (np.int32(42), 1262), (np.int32(44), 1251), (np.int32(48), 1235), (np.int32(17), 1174), (np.int32(43), 1133), (np.int32(47), 1121), (np.int32(35), 1113), (np.int32(37), 1109), (np.int32(4), 1103), (np.int32(16), 1091), (np.int32(33), 1091), (np.int32(7), 1066), (np.int32(22), 1055), (np.int32(15), 1042), (np.int32(36), 1038), (np.int32(31), 1027), (np.int32(9), 1009), (np.int32(5), 991), (np.int32(6), 969), (np.int32(27), 953), (np.int32(10), 915), (np.int32(1), 905), (np.int32(2), 899), (np.int32(0), 894), (np.int32(21), 844), (np.int32(34), 835), (np.int32(19), 820), (np.int32(38), 807), (np.int32(28), 807), (np.int32(3), 796), (np.int32(14), 794), (np.int32(39), 736), (np.int32(12), 661), (np.int32(18), 639), (np.int32(45), 607), (np.int32(13), 594), (np.int32(26), 5

In [41]:
# This cluster seems to contain mostly superhero movies/sci-fi movies
test_cluster = 26

new_conn.execute("""
    SELECT m.title
    FROM movies m
    JOIN movie_clusters c ON m.id = c.movie_id
    WHERE c.cluster_id = ?
    LIMIT 100
""", (test_cluster,)).fetchall()

[('Star Wars',),
 ('The Dark Knight',),
 ('Batman Begins',),
 ('Catwoman',),
 ('Spider-Man',),
 ('Spider-Man 2',),
 ('Spider-Man 3',),
 ('Austin Powers in Goldmember',),
 ('Superman Returns',),
 ('The Incredible Hulk',),
 ('Iron Man',),
 ('Captain America: The First Avenger',),
 ('Transformers',),
 ('Star Wars: Episode I - The Phantom Menace',),
 ('Hulk',),
 ('The Amazing Spider-Man',),
 ('My Name Is Bruce',),
 ('Fantastic Four: Rise of the Silver Surfer',),
 ('X-Men Origins: Wolverine',),
 ('American Splendor',),
 ('My Super Ex-Girlfriend',),
 ('The Spirit',),
 ('Transformers: Revenge of the Fallen',),
 ('The League of Extraordinary Gentlemen',),
 ('Hancock',),
 ('Daredevil',),
 ('The Incredibles',),
 ('Robots',),
 ('Iron Man 2',),
 ('Thor',),
 ('Sky High',),
 ('Superhero Movie',),
 ('Spy Kids 3-D: Game Over',),
 ('Jimmy Neutron: Boy Genius',),
 ('Punisher: War Zone',),
 ('Hellboy Animated: Blood and Iron',),
 ('Superman: Doomsday',),
 ('The Invincible Iron Man',),
 ('Batman: Gotham K